In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
#cleaned data
df = pd.read_csv('../data/processed/housing_data_cleaned.csv', parse_dates=['date'])
print("Loaded data shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print("Unique zipcodes:", df['zipcode'].nunique())
df.head()

Loaded data shape: (260383, 43)
Date range: 2012-03-31 00:00:00 to 2023-12-31 00:00:00
Unique zipcodes: 1888


,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full,month,quarter,list_sale_gap,ppsf_gap
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,...,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta,3,1,0.785928,0.711267
1,2012-04-30,61870.0,245000.0,40.723982,130.528256,22.0,29.0,56.0,69.0,89.5,...,460.0,5408.0,5408.0,2492.0,202421.064584,Atlanta-Sandy Springs-Alpharetta,4,2,0.747469,0.688006
2,2012-05-31,125500.0,217450.0,63.913043,119.919216,24.0,40.0,63.0,60.0,144.5,...,460.0,5408.0,5408.0,2492.0,202681.309539,Atlanta-Sandy Springs-Alpharetta,5,2,0.422856,0.467033
3,2012-06-30,153000.0,189900.0,81.598080,105.617353,34.0,46.0,50.0,57.0,126.0,...,460.0,5408.0,5408.0,2492.0,202998.603897,Atlanta-Sandy Springs-Alpharetta,6,2,0.194313,0.227418
4,2012-07-31,165500.0,154000.0,81.598080,83.921175,39.0,49.0,42.0,50.0,80.0,...,460.0,5408.0,5408.0,2492.0,203781.903446,Atlanta-Sandy Springs-Alpharetta,7,3,-0.074675,0.027682


In [23]:
df = pd.read_csv('../data/processed/housing_data_cleaned.csv', parse_dates=['date'])
print("Loaded data shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print("Unique zipcodes:", df['zipcode'].nunique())
df.head()

Loaded data shape: (302443, 43)
Date range: 2012-03-31 00:00:00 to 2023-12-31 00:00:00
Unique zipcodes: 2205


,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,...,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full,month,quarter,list_sale_gap,ppsf_gap
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,...,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta,3,1,0.785928,0.711267
1,2012-04-30,61870.0,245000.0,40.723982,130.528256,22.0,29.0,56.0,69.0,89.5,...,460.0,5408.0,5408.0,2492.0,202421.064584,Atlanta-Sandy Springs-Alpharetta,4,2,0.747469,0.688006
2,2012-05-31,125500.0,217450.0,63.913043,119.919216,24.0,40.0,63.0,60.0,144.5,...,460.0,5408.0,5408.0,2492.0,202681.309539,Atlanta-Sandy Springs-Alpharetta,5,2,0.422856,0.467033
3,2012-06-30,153000.0,189900.0,81.598080,105.617353,34.0,46.0,50.0,57.0,126.0,...,460.0,5408.0,5408.0,2492.0,202998.603897,Atlanta-Sandy Springs-Alpharetta,6,2,0.194313,0.227418
4,2012-07-31,165500.0,154000.0,81.598080,83.921175,39.0,49.0,42.0,50.0,80.0,...,460.0,5408.0,5408.0,2492.0,203781.903446,Atlanta-Sandy Springs-Alpharetta,7,3,-0.074675,0.027682


In [24]:
# sorting by the zipcode and date
df = df.sort_values(['zipcode', 'date'])

# finding future price change (maybe we can do 6 months ahead)
df['future_price_6m'] = df.groupby('zipcode')['median_sale_price'].shift(-6)
df['price_change_6m_pct'] = (df['future_price_6m'] - df['median_sale_price']) / df['median_sale_price'] * 100

# downturn target variable (1 if price drops more than 10% in 6 months)
df['downturn_risk'] = (df['price_change_6m_pct'] < -10).astype(int)

# number of downturns and rate
print(f"Downturn rate: {df['downturn_risk'].mean()*100:.2f}%")
print(f"Number of downturn events: {df['downturn_risk'].sum()}")


Downturn rate: 13.44%
Number of downturn events: 40647


In [25]:
## PRICE TREND FEATURES
# rolling avg
for window in [3, 6, 12]:
    df[f'price_ma_{window}m'] = df.groupby('zipcode')['median_sale_price'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )

# month over month (price momentum??)
df['price_mom'] = df.groupby('zipcode')['median_sale_price'].pct_change() * 100

# volatility
for window in [3, 6]:
    df[f'price_volatility_{window}m'] = df.groupby('zipcode')['median_sale_price'].transform(
        lambda x: x.rolling(window, min_periods=2).std()
    )

In [26]:
## MARKET HEALTH FEATURES
# sale to list ratio
if 'avg_sale_to_list' in df.columns:
    df['stl_momentum'] = df.groupby('zipcode')['avg_sale_to_list'].pct_change() * 100

# inventory turnover
if 'homes_sold' in df.columns and 'inventory' in df.columns:
    df['inventory_turnover'] = df['homes_sold'] / (df['inventory'] + 1)

# days on market trend
if 'median_dom' in df.columns:
    df['dom_trend'] = df.groupby('zipcode')['median_dom'].pct_change() * 100

In [27]:
# past prices from 3,6,12 months ago?
# okay so .shift lag moves data up or down a certain number of rows so it allows up to look at the prices from that month relative
# to whatever row yo're on
for lag in [3, 6, 12]:
    df[f'price_lag_{lag}m'] = df.groupby('zipcode')['median_sale_price'].shift(lag)
    df[f'inventory_lag_{lag}m'] = df.groupby('zipcode')['inventory'].shift(lag)
    df[f'sales_lag_{lag}m'] = df.groupby('zipcode')['homes_sold'].shift(lag)


In [28]:
# getting rid of any NaNs now from rolling computations
initial_len = len(df)
df = df.dropna()
print(f"Final dataset shape: {df.shape}")

Final dataset shape: (262780, 64)


In [29]:
df.to_csv('../data/processed/housing_data_engineered.csv', index=False)

In [30]:
df_yearly = df.groupby('year')['downturn_risk'].mean()
print(df_yearly)

year
2013    0.189428
2014    0.166882
2015    0.153367
2016    0.131470
2017    0.126860
2018    0.143402
2019    0.126306
2020    0.083475
2021    0.095579
2022    0.194235
2023    0.113249
Name: downturn_risk, dtype: float64
